# 📈 Pocket Option OTC High-Frequency Tick Data Exploration
Welcome to the interactive exploration notebook for the **Pocket Option OTC Tick Dataset**.

This notebook demonstrates how to:
1. Load high-speed compressed **Parquet** tick streams into Pandas.
2. Inspect **gap-aware continuous trading sessions** (`session_id`).
3. Resample microsecond-precision ticks into **5-second & 1-minute OHLCV Candlesticks**.
4. Analyze **Sigmoid Liquidity %** vs **Volatility** distributions.
5. Run a vectorized **60-Second Momentum Trading Strategy** backtest.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configure clean plotting style
plt.style.use("seaborn-v0_8-darkgrid" if "seaborn-v0_8-darkgrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (14, 6)
print("Libraries successfully imported!")


## 1. Load Parquet Tick Stream
Parquet files load in milliseconds and preserve exact datatypes and sub-second timestamps.

In [ ]:
# Load Sample File (or replace with any full pair in data/parquet/)
sample_path = Path("data/samples/EURUSD_otc_sample.parquet")
if not sample_path.exists():
    sample_path = list(Path("data/parquet").glob("*EURUSD*.parquet"))[0]

df = pd.read_parquet(sample_path)
print(f"Loaded {len(df):,} ticks from {sample_path.name}")
df.head()


## 2. Inspect Gap-Aware Sessions (`session_id`)
Gaps larger than 60 seconds (broker downtime, sleep epochs) are automatically assigned a new `session_id`. This prevents cross-gap indicator leakage during backtesting.

In [ ]:
session_summary = df.groupby("session_id").agg(
    start_time=("datetime_utc", "first"),
    end_time=("datetime_utc", "last"),
    ticks=("price", "count"),
    avg_tpm=("ticks_per_min", "mean"),
    avg_liquidity=("sigmoid_liquidity", "mean"),
    avg_volatility=("volatility_score", "mean"),
).reset_index()

print(f"Total Continuous Trading Sessions: {len(session_summary)}")
session_summary.head(10)


## 3. Resample Sub-Second Ticks to 5-Second OHLCV Candles

In [ ]:
# Select Session 1 for clean analysis
s1 = df[df["session_id"] == 1].copy()
s1["dt"] = pd.to_datetime(s1["timestamp"], unit="s", utc=True)
s1.set_index("dt", inplace=True)

# 5-Second Resampling
candles_5s = s1["price"].resample("5s").ohlc()
candles_5s["volume_ticks"] = s1["price"].resample("5s").count()
candles_5s["avg_liquidity"] = s1["sigmoid_liquidity"].resample("5s").mean()
candles_5s["avg_volatility"] = s1["volatility_score"].resample("5s").mean()
candles_5s.dropna(inplace=True)

# Plot Price & Volume
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(candles_5s.index, candles_5s["close"], label="5s Close Price", color="#0284c7", lw=1.5)
ax1.set_title("EUR/USD OTC — 5-Second Price Action (Session 1)", fontsize=14, fontweight="bold")
ax1.set_ylabel("Price")
ax1.legend(loc="upper left")

ax2.bar(candles_5s.index, candles_5s["volume_ticks"], width=pd.Timedelta(seconds=4), color="#10b981", alpha=0.7, label="Tick Volume / 5s")
ax2.set_ylabel("Tick Count")
ax2.set_xlabel("UTC Time")
ax2.legend(loc="upper left")
plt.tight_layout()
plt.show()


## 4. Sigmoid Liquidity Corridor & Volatility Analysis
Pocket Option tick velocity dynamically shifts. We use a Sigmoid function centered at 120 ticks/min to calculate a 0–100% liquidity score.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Liquidity Histogram
ax1.hist(df["sigmoid_liquidity"], bins=30, color="#06b6d4", edgecolor="black", alpha=0.8)
ax1.axvline(30, color="#f59e0b", linestyle="--", label="Min Gate Threshold (30%)")
ax1.axvline(70, color="#ef4444", linestyle="--", label="Max Gate Threshold (70%)")
ax1.set_title("Sigmoid Liquidity % Distribution", fontsize=12, fontweight="bold")
ax1.set_xlabel("Liquidity Score (%)")
ax1.set_ylabel("Tick Count")
ax1.legend()

# Volatility Histogram
ax2.hist(df["volatility_score"], bins=30, color="#8b5cf6", edgecolor="black", alpha=0.8)
ax2.set_title("Volatility Score Distribution", fontsize=12, fontweight="bold")
ax2.set_xlabel("Volatility Score (%)")
ax2.set_ylabel("Tick Count")

plt.tight_layout()
plt.show()


## 5. Vectorized 60-Second Binary Options Strategy Backtest
Let's simulate a classic Momentum Breakout strategy:
- **CALL Condition:** Price breaks above 60s high AND Liquidity in [30%, 70%].
- **PUT Condition:** Price breaks below 60s low AND Liquidity in [30%, 70%].
- **Expiry:** 60 Seconds.
- **Payout:** 92% (Standard Pocket Option OTC payout).

In [ ]:
# Resample to 1-second ticks for precise 60s trade settlement
sec_df = s1["price"].resample("1s").last().ffill().to_frame(name="price")
sec_df["rolling_high_60s"] = sec_df["price"].rolling(60).max().shift(1)
sec_df["rolling_low_60s"] = sec_df["price"].rolling(60).min().shift(1)
sec_df["future_price_60s"] = sec_df["price"].shift(-60)
sec_df.dropna(inplace=True)

# Signals
call_signal = sec_df["price"] > sec_df["rolling_high_60s"]
put_signal = sec_df["price"] < sec_df["rolling_low_60s"]

# Outcomes
call_wins = (sec_df["future_price_60s"] > sec_df["price"]) & call_signal
call_losses = (sec_df["future_price_60s"] <= sec_df["price"]) & call_signal

total_trades = call_signal.sum()
wins = call_wins.sum()
win_rate = (wins / total_trades) * 100.0 if total_trades > 0 else 0

print("--- 60-Second Momentum Strategy Backtest Results ---")
print(f"Total Trades Executed: {total_trades:,}")
print(f"Wins: {wins:,} | Losses: {call_losses.sum():,}")
print(f"Raw Win Rate: {win_rate:.2f}%")
payout = 0.92
ev = ((win_rate / 100.0) * payout) - (((100.0 - win_rate) / 100.0) * 1.0)
print(f"Expected Value / Trade (@ 92% payout): {ev:.3f} R")
